# Sparse Feature Tracker — Exploratory Analysis

This notebook walks through the core ideas of the project interactively:
1. Load configuration and prompt families
2. Inspect sample prompt families
3. Run a mini feature extraction on a handful of prompts
4. Examine top-k features
5. Compute a small Jaccard overlap matrix
6. Display an inline figure

Run the full pipeline scripts for complete results.

In [ ]:
import sys, os
# Ensure the project root is on the path when running from notebooks/
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print('Imports OK')

## 1. Load Configuration

In [ ]:
from src.utils.io import load_config
from src.utils.seed import set_seed

config = load_config('../configs/default.yaml')
set_seed(config['seed'])

print('Configuration loaded:')
for section, values in config.items():
    print(f'  [{section}]')
    if isinstance(values, dict):
        for k, v in values.items():
            print(f'    {k}: {v}')
    else:
        print(f'    {values}')

## 2. Build and Inspect Prompt Families

Each **prompt family** groups paraphrased prompts that share the same correct answer, plus distractor prompts used as negative controls.

In [ ]:
from src.data.prompt_families import build_factual_recall_families

families = build_factual_recall_families()
print(f'Total families: {len(families)}')
print(f'Total paraphrase prompts: {sum(len(f.paraphrases) for f in families)}')
print(f'Total distractor prompts: {sum(len(f.incorrect_prompts) for f in families)}')

# Show family IDs
for f in families:
    print(f'  {f.family_id:<30} correct_answer={f.correct_answer!r}')

In [ ]:
# Inspect one family in detail
example = next(f for f in families if f.family_id == 'capital_france')

print(f'Family: {example.family_id}')
print(f'Topic: {example.topic}')
print(f'Correct answer: {example.correct_answer!r}')
print()
print('Paraphrases:')
for i, p in enumerate(example.paraphrases, 1):
    print(f'  {i}. {p!r}')
print()
print('Distractor prompts:')
for i, p in enumerate(example.incorrect_prompts, 1):
    print(f'  {i}. {p!r}')

## 3. Load Model and SAE

We load GPT-2 small and a pretrained Sparse Autoencoder trained on the layer-8 residual stream.

In [ ]:
from src.models.load_model import load_model_and_tokenizer
from src.features.load_sae import load_pretrained_sae

device = config['model']['device']
model, tokenizer = load_model_and_tokenizer(config['model']['name'], device)

sae, sae_cfg = load_pretrained_sae(
    release=config['sae']['release'],
    sae_id=config['sae']['id'],
    device=device,
)

n_features = sae_cfg.get('d_sae') or sae_cfg.get('n_features', 'unknown')
print(f'SAE n_features: {n_features}')
print(f'SAE config keys: {list(sae_cfg.keys())}')

## 4. Mini Feature Extraction

Run feature extraction on 3 paraphrases from the France capital family to see which sparse features activate.

In [ ]:
from src.features.extract_features import FeatureExtractor

extractor = FeatureExtractor(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    layer_idx=config['sae']['layer'],
    device=device,
    top_k=config['features']['top_k'],
)

# Pick 3 prompts from the France capital family
mini_prompts = example.paraphrases[:3]
results = extractor.extract_for_prompts(mini_prompts)

print('Extraction complete.')
for r in results:
    print(f'\nPrompt: {r["prompt"]!r}')
    print(f'  Top-5 feature indices: {r["top_k_indices"][:5]}')
    print(f'  Top-5 feature values : {[f"{v:.3f}" for v in r["top_k_values"][:5]]}')

## 5. Next-Token Predictions

Check what the model actually predicts as the next token for each prompt.

In [ ]:
print(f'Correct answer: {example.correct_answer!r}\n')
for prompt in mini_prompts:
    pred = extractor.get_next_token_prediction(prompt)
    match = pred.strip().lower() == example.correct_answer.strip().lower()
    status = '✓' if match else '✗'
    print(f'{status}  Prompt: {prompt!r}')
    print(f'   Predicted: {pred!r}')

## 6. Pairwise Jaccard Overlap Matrix

Compute the 3×3 pairwise Jaccard overlap between the top-k feature sets of our three prompts.  High values mean the same features are active — evidence of stable, prompt-invariant representations.

In [ ]:
from src.features.feature_stats import (
    compute_family_overlap_matrix,
    compute_mean_family_stability,
)

overlap_matrix = compute_family_overlap_matrix(results)
stability = compute_mean_family_stability(results)

print('Pairwise Jaccard overlap matrix:')
df_mat = pd.DataFrame(
    overlap_matrix,
    index=[f'P{i+1}' for i in range(len(results))],
    columns=[f'P{i+1}' for i in range(len(results))],
)
print(df_mat.round(3).to_string())
print(f'\nMean pairwise stability: {stability:.3f}')

## 7. Inline Visualisation

Plot the overlap matrix as a heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))

truncated_labels = [
    r['prompt'][:30] + '...' if len(r['prompt']) > 30 else r['prompt']
    for r in results
]

sns.heatmap(
    overlap_matrix,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=[f'P{i+1}' for i in range(len(results))],
    yticklabels=truncated_labels,
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Jaccard'},
    linewidths=0.5,
)
ax.set_title(
    f'Top-{config["features"]["top_k"]} Feature Overlap\n'
    f'(mean stability = {stability:.2f})',
    fontsize=11,
    fontweight='bold',
)
fig.tight_layout()
plt.show()

## 8. Top Features by Frequency

Across the 3 prompts, which features appear most often in the top-k set?

In [ ]:
from src.features.feature_stats import rank_features_by_frequency

freq_df = rank_features_by_frequency(results)
print('Top 10 features by frequency:')
print(freq_df.head(10).to_string(index=False))

## 9. Quick Consistency Check Across All Families

Run a lightweight consistency evaluation on the first 3 families only (to keep this notebook fast). Run `scripts/run_consistency_eval.py` for the full evaluation.

In [ ]:
from src.evaluation.consistency import evaluate_paraphrase_consistency

# Use only the first 3 families for speed
mini_families = families[:3]

consistency_df = evaluate_paraphrase_consistency(
    extractor=extractor,
    families=mini_families,
    top_k=config['features']['top_k'],
)
print(consistency_df.to_string(index=False, float_format='{:.3f}'.format))

## 10. Inline Consistency Bar Chart

In [ ]:
if not consistency_df.empty:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    x = range(len(consistency_df))
    ax.bar(
        x,
        consistency_df['mean_overlap'],
        yerr=consistency_df['std_overlap'],
        capsize=5,
        color=sns.color_palette('Blues_d', len(consistency_df))[::-1],
        edgecolor='white',
        error_kw={'elinewidth': 1.5, 'ecolor': '#555'},
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        consistency_df['family_id'].str.replace('_', ' ').str.title(),
        rotation=20, ha='right',
    )
    ax.set_ylabel('Mean Jaccard Overlap')
    ax.set_title('Feature Stability (mini eval)', fontweight='bold')
    ax.set_ylim(0, 1.05)
    fig.tight_layout()
    plt.show()
else:
    print('No consistency data to plot.')

## Summary

This notebook demonstrated the full analysis loop at a small scale:

| Step | What we did |
|------|-------------|
| Data | Built 18 factual-recall prompt families |
| Model | Loaded GPT-2 small with a layer-8 residual stream hook |
| SAE | Encoded activations into a sparse feature space |
| Analysis | Computed pairwise Jaccard overlap across paraphrases |
| Result | Feature sets are partially stable across semantically equivalent prompts |

For the full experiment, run the pipeline scripts in order:
```bash
python scripts/run_dataset.py
python scripts/run_feature_extraction.py
python scripts/run_consistency_eval.py
python scripts/run_behavior_eval.py
python scripts/run_plots.py
```